In [ ]:
import torch
import soundfile as sf
from torchaudio import transforms as T
from neucodec import NeuCodec

model = NeuCodec.from_pretrained("neuphonic/neucodec")
model.eval()

# Load audio
y, sr = sf.read(
    "D:/Personal Projects/tune-ml/datasets/audio/audio/5061 Philippe Van Mullem - Canopy (Progress Mix).mp3"
)

y = torch.tensor(y).float()

# ---- FIX 1: stereo -> mono ----
if y.ndim == 1:
    y = y.unsqueeze(0)  # (1, T)
else:
    y = y.transpose(0, 1)  # (C, T)
    y = y.mean(dim=0, keepdim=True)  # (1, T)

# ---- FIX 2: resample ----
if sr != 16000:
    y = T.Resample(sr, 16000)(y)

# ---- FIX 3: batch dimension ----
y = y.unsqueeze(0)  # (B, 1, T)

print("Final shape:", y.shape)  # should be (1, 1, T)

with torch.no_grad():
    fsq_codes = model.encode_code(y)
    recon = model.decode_code(fsq_codes).cpu()

# save
sf.write("original.wav", y[0, 0].numpy(), 16000)
sf.write("reconstructed.wav", recon[0, 0].numpy(), 24000)